In [2]:
%pip install -q "mlflow==3.10.1" sagemaker-mlflow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import boto3
import mlflow

AWS_REGION = "ap-south-1"

MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-south-1:"
    "812224290846:mlflow-app/app-QGCZHTATSVV6"
)

EXPECTED_ARTIFACT_STORE = (
    "s3://krushang-beverage-ml-2026/mlflow-artifacts/"
)

SAGEMAKER_JOB_NAME = (
    "beverage-xgboost-production-20260904074049"
)

MODEL_ARTIFACT_URI = (
    "s3://krushang-beverage-ml-2026/models/"
    "beverage-xgboost-production-20260904074049/"
    "output/model.tar.gz"
)

print("MLflow client version:", mlflow.__version__)
print("Region:", AWS_REGION)
print("MLflow App:", MLFLOW_APP_ARN)

MLflow client version: 3.10.1
Region: ap-south-1
MLflow App: arn:aws:sagemaker:ap-south-1:812224290846:mlflow-app/app-QGCZHTATSVV6


In [2]:
sm = boto3.client(
    "sagemaker",
    region_name=AWS_REGION
)

app = sm.describe_mlflow_app(
    Arn=MLFLOW_APP_ARN
)

print("Name:", app["Name"])
print("Status:", app["Status"])
print("MLflow version:", app["MlflowVersion"])
print("Artifact store:", app["ArtifactStoreUri"])
print("Model registration:", app["ModelRegistrationMode"])

Name: beverage-mlflow-tracking
Status: Created
MLflow version: 3.10.1
Artifact store: s3://krushang-beverage-ml-2026/mlflow-artifacts/
Model registration: AutoModelRegistrationDisabled


In [3]:
assert app["Status"] in ["Created", "Updated"]
assert app["ArtifactStoreUri"].rstrip("/") == (
    EXPECTED_ARTIFACT_STORE.rstrip("/")
)
assert app["MlflowVersion"] == "3.10.1"

print("✅ MLflow App configuration validated")

✅ MLflow App configuration validated


In [4]:
mlflow.set_tracking_uri(MLFLOW_APP_ARN)

print(
    "Tracking URI:",
    mlflow.get_tracking_uri()
)

Tracking URI: arn:aws:sagemaker:ap-south-1:812224290846:mlflow-app/app-QGCZHTATSVV6


In [5]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiments = client.search_experiments()

print("✅ Connected to SageMaker Managed MLflow")
print("Experiments found:", len(experiments))

for exp in experiments[:10]:
    print(
        exp.experiment_id,
        "|",
        exp.name
    )

✅ Connected to SageMaker Managed MLflow
Experiments found: 1
0 | Default


In [6]:
import boto3
import io
import json
import tarfile
from urllib.parse import urlparse

s3 = boto3.client(
    "s3",
    region_name=AWS_REGION
)

# --------------------------------------------------
# Read actual SageMaker Training Job
# --------------------------------------------------

job = sm.describe_training_job(
    TrainingJobName=SAGEMAKER_JOB_NAME
)

channels = {
    channel["ChannelName"]:
    channel["DataSource"]["S3DataSource"]["S3Uri"]
    for channel in job["InputDataConfig"]
}

actual_model_uri = (
    job["ModelArtifacts"]["S3ModelArtifacts"]
)

print("Training status:", job["TrainingJobStatus"])
print("Instance:", job["ResourceConfig"]["InstanceType"])
print("Training data:", channels["training"])
print("Config:", channels["config"])
print("Model artifact:", actual_model_uri)


# --------------------------------------------------
# Read training_metadata.json directly from model.tar.gz
# --------------------------------------------------

parsed = urlparse(actual_model_uri)

artifact_bucket = parsed.netloc
artifact_key = parsed.path.lstrip("/")

artifact_bytes = s3.get_object(
    Bucket=artifact_bucket,
    Key=artifact_key
)["Body"].read()

with tarfile.open(
    fileobj=io.BytesIO(artifact_bytes),
    mode="r:gz"
) as tar:

    metadata_file = tar.extractfile(
        "training_metadata.json"
    )

    metadata = json.loads(
        metadata_file.read().decode("utf-8")
    )

print("\nProduction metadata:")
print(json.dumps(metadata, indent=2))

Training status: Completed
Instance: ml.m5.large
Training data: s3://krushang-beverage-ml-2026/processed/cleaned_survey_results.csv
Config: s3://krushang-beverage-ml-2026/evaluation/xgboost_best_params.json
Model artifact: s3://krushang-beverage-ml-2026/models/beverage-xgboost-production-20260904074049/output/model.tar.gz

Production metadata:
{
  "model": "XGBoost",
  "purpose": "production_refit",
  "training_rows": 29956,
  "feature_count": 27,
  "hyperparameters": {
    "n_estimators": 700,
    "max_depth": 6,
    "learning_rate": 0.12034778187740461,
    "subsample": 0.9350139863013668,
    "colsample_bytree": 0.9552153727216537,
    "min_child_weight": 5,
    "gamma": 0.13876309692419167,
    "reg_alpha": 0.03870751721089646,
    "reg_lambda": 1.9028977942385588
  },
  "xgboost_version": "2.1.4",
  "sklearn_version": "1.4.2",
  "trained_at_utc": "2026-09-04T07:42:52.893333+00:00",
  "official_holdout_metrics": {
    "accuracy": 0.9246,
    "macro_f1": 0.9236,
    "ordinal_mae": 0

In [7]:
EXPERIMENT_NAME = (
    "beverage-price-prediction-production"
)

experiment = mlflow.set_experiment(
    EXPERIMENT_NAME
)

print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

2026/09/04 08:52:41 INFO mlflow.tracking.fluent: Experiment with name 'beverage-price-prediction-production' does not exist. Creating a new experiment.


Experiment: beverage-price-prediction-production
Experiment ID: 1


In [8]:
RUN_NAME = (
    "xgboost-sagemaker-production-"
    "20260904074049"
)

# --------------------------------------------------
# Parameters
# --------------------------------------------------

run_params = {
    **metadata["hyperparameters"],

    "training_rows":
        metadata["training_rows"],

    "feature_count":
        metadata["feature_count"],

    "xgboost_version":
        metadata["xgboost_version"],

    "sklearn_version":
        metadata["sklearn_version"],

    "instance_type":
        job["ResourceConfig"]["InstanceType"],

    "instance_count":
        job["ResourceConfig"]["InstanceCount"],
}


# --------------------------------------------------
# Metrics
# --------------------------------------------------

run_metrics = {
    "accuracy":
        metadata["official_holdout_metrics"]["accuracy"],

    "macro_f1":
        metadata["official_holdout_metrics"]["macro_f1"],

    "ordinal_mae":
        metadata["official_holdout_metrics"]["ordinal_mae"],
}

if "TrainingTimeInSeconds" in job:
    run_metrics["training_time_seconds"] = (
        job["TrainingTimeInSeconds"]
    )

if "BillableTimeInSeconds" in job:
    run_metrics["billable_time_seconds"] = (
        job["BillableTimeInSeconds"]
    )


# --------------------------------------------------
# Lineage
# --------------------------------------------------

lineage = {
    "training_job_name":
        SAGEMAKER_JOB_NAME,

    "training_job_arn":
        job["TrainingJobArn"],

    "training_data_uri":
        channels["training"],

    "hyperparameter_config_uri":
        channels["config"],

    "model_artifact_uri":
        actual_model_uri,

    "instance_type":
        job["ResourceConfig"]["InstanceType"],

    "training_status":
        job["TrainingJobStatus"],

    "training_start_time":
        job["TrainingStartTime"].isoformat(),

    "training_end_time":
        job["TrainingEndTime"].isoformat(),
}


# --------------------------------------------------
# Create MLflow Run
# --------------------------------------------------

with mlflow.start_run(
    run_name=RUN_NAME
) as active_run:

    mlflow.log_params(run_params)

    mlflow.log_metrics(run_metrics)

    mlflow.set_tags({
        "project":
            "Beverage Price Prediction",

        "algorithm":
            "XGBoost",

        "stage":
            "production_refit",

        "aws_region":
            AWS_REGION,

        "compute_service":
            "Amazon SageMaker Training",

        "sagemaker_training_job":
            SAGEMAKER_JOB_NAME,

        "model_artifact_uri":
            actual_model_uri,

        "tracking_type":
            "retrospective_completed_training_run",
    })

    mlflow.log_dict(
        metadata,
        "metadata/training_metadata.json"
    )

    mlflow.log_dict(
        lineage,
        "lineage/sagemaker_training_job.json"
    )

    RUN_ID = active_run.info.run_id


print("✅ MLflow run logged")
print("Run ID:", RUN_ID)
print("Experiment:", EXPERIMENT_NAME)

🏃 View run xgboost-sagemaker-production-20260904074049 at: https://mlflow.sagemaker.ap-south-1.app.aws/#/experiments/1/runs/d49f7a3220c24123bbb1a472fe3016d0
🧪 View experiment at: https://mlflow.sagemaker.ap-south-1.app.aws/#/experiments/1
✅ MLflow run logged
Run ID: d49f7a3220c24123bbb1a472fe3016d0
Experiment: beverage-price-prediction-production


In [9]:
logged_run = client.get_run(RUN_ID)

print("Run ID:", logged_run.info.run_id)
print("Status:", logged_run.info.status)

print("\nPARAMETERS")
for key, value in logged_run.data.params.items():
    print(f"{key}: {value}")

print("\nMETRICS")
for key, value in logged_run.data.metrics.items():
    print(f"{key}: {value}")

print("\nIMPORTANT TAGS")
for key in [
    "project",
    "algorithm",
    "stage",
    "aws_region",
    "compute_service",
    "sagemaker_training_job",
    "tracking_type",
]:
    print(
        f"{key}:",
        logged_run.data.tags.get(key)
    )

Run ID: d49f7a3220c24123bbb1a472fe3016d0
Status: FINISHED

PARAMETERS
n_estimators: 700
max_depth: 6
learning_rate: 0.12034778187740461
subsample: 0.9350139863013668
colsample_bytree: 0.9552153727216537
min_child_weight: 5
gamma: 0.13876309692419167
reg_alpha: 0.03870751721089646
reg_lambda: 1.9028977942385588
training_rows: 29956
feature_count: 27
xgboost_version: 2.1.4
sklearn_version: 1.4.2
instance_type: ml.m5.large
instance_count: 1

METRICS
accuracy: 0.9246
macro_f1: 0.9236
ordinal_mae: 0.0754
training_time_seconds: 104.0
billable_time_seconds: 104.0

IMPORTANT TAGS
project: Beverage Price Prediction
algorithm: XGBoost
stage: production_refit
aws_region: ap-south-1
compute_service: Amazon SageMaker Training
sagemaker_training_job: beverage-xgboost-production-20260904074049
tracking_type: retrospective_completed_training_run
